# 07 — 비필수로 오분류된 필수 유전자의 적합도 예측 분포 (`results/fitstat_essfn`)

웹 서비스는 필수로 판정된 유전자만 적합도 칸을 비우므로, **분류기가 놓친 필수 유전자(위음성, FN)는 적합도 예측을 받는다.**
그런데 적합도 모델은 비필수 유전자로만 학습 · 평가했다(03 · 04 노트북의 `finite_only=True`). 이 노트북은 교차검증 조건 그대로
**필수 유전자를 포함한 전 샘플**에 적합도 모델을 적용해, FN 필수 유전자와 제대로 비필수로 판정된 비필수 유전자(TN)의 예측 적합도 분포를 비교한다.

| 그룹 | 정의 | 그림 색 |
|---|---|---|
| **FN 필수** | 실제 필수 (`fit_mean = -inf`) · 분류 판정 비필수 (`conf_ess < 0.5`) | 파랑 |
| **TN 비필수** | 실제 비필수 (`fit_mean` 수치) · 분류 판정 비필수 | 녹색 |

**모델과 판정**
- 분류 — 서비스 모델 `esscls_ablation/poolesm-noseq` 의 교차검증 **보정** 확률 `conf_ess` (`pred_calib.npz`), 판정 = `conf_ess ≥ 0.5` (서비스와 같다)
- 적합도 — 서비스 모델 `fitpred_ablation/noseq` 의 **폴드별 모델** (`fold##_<genus>/inner##/model.pt`). 시험 genus 를 학습에서 뺀 모델이므로 처음 보는 genus 에 대한 예측이다
- 시험 genus 마다 내부 폴드 모델이 3개씩 있다. 유전자별로 **3개 모델의 평균**을 쓴다 (`conf_ess` 도 3개 평균 후 판정, 적합도 `mu` 도 3개 평균)
- 예측 적합도 = 미니 MLP 1,000개 예측의 평균 $\mu$ (균주 내 z-점수, 0 = 균주 평균, 음수 = 생장 저하)

**입력 재현** — 적합도 모델의 입력은 ESM 토큰 · ProteomeLM · Biopython 특성 · 문맥 특성 3종이다. 문맥 특성은 내부 폴드마다 참조 균주가 달라
(시험 · 검증 · 자기 genus 제외) 폴드마다 다시 만들고, **표준화 기준은 그 내부 폴드의 원래 훈련 샘플**(적합도 수치가 있고 시험 · 검증 genus 밖)로 잡는다.
문맥 특성은 유전자마다 따로 계산되므로, 이렇게 하면 비필수 유전자의 입력은 학습 때와 같고 필수 유전자도 같은 규칙으로 입력이 만들어진다.
→ **비필수 유전자의 예측이 저장된 교차검증 예측(`pred.npz`)과 같은지** 폴드마다 확인한다 (6절 `check` 표).

**통계** — 예측 적합도 분포는 한쪽으로 치우치고 꼬리가 길어 정규성 가정이 맞지 않는다. 그래서 분포 가정이 필요 없는 **Mann–Whitney U 검정**
(윌콕슨 순위합 검정, 두 독립 표본, 양측)으로 비교한다. 효과 크기는 $P(\text{FN} < \text{TN})$ — 무작위로 뽑은 FN 유전자의 예측 적합도가
무작위로 뽑은 TN 유전자보다 낮을 확률이다 (동점은 절반, 0.5 = 차이 없음). 박스플롯은 두 박스를 잇는 선 위에 genus 별 p 값의 별표를 단다
(`***` < 0.001, `**` < 0.01, `*` < 0.05, `ns`).

**로컬 PC 실행** (GTX 1050 Ti 4 GB · 16코어 · RAM 34 GB · ESM 토큰 h5 165 GB 가 HDD)

| 단계 | 시간 | 최적화 |
|---|---|---|
| Biopython 특성 · 직교상동체 목록 | 처음 한 번 (`work/` 에 저장) | GPU 행렬곱 |
| 폴드별 데이터셋 (84개 = 28 genus × 3) | genus 당 수십 초 | — |
| ESM 토큰 읽기 + 추론 | HDD 읽기가 병목 (165 GB, 약 190 MB/s) | 균주 단위 **연속 구간 읽기**, 다음 묶음을 **백그라운드 스레드로 미리 읽기**, **핀 메모리 → 비동기 GPU 복사**, 길이순 배치, 토큰을 한 번 읽어 내부 폴드 모델 3개에 같이 넣기 |

- `torch.compile` 은 쓰지 않는다. Windows 에는 기본 백엔드(Inductor)가 쓰는 Triton 이 없고, 병목이 디스크 읽기라 이득도 없다
- 혼합정밀도도 쓰지 않는다 (fp32). GTX 1050 Ti (Pascal) 는 bf16 을 지원하지 않고 fp16 연산이 느리다. 학습 때는 bf16 이었으므로 예측값이 소수점 아래에서 조금 다를 수 있다
- genus 마다 결과를 `work/pred_##_<genus>.npz` 로 저장하므로 중간에 멈춰도 이어서 돌릴 수 있다

**산출물** (`results/fitstat_essfn/`) — `pred_all.tsv` (유전자별 `conf_ess` · 판정 · 그룹 · `mu` · `sd`), `check.csv` (재현 확인),
`stats.csv` (genus 별 · 전체 평균 · 중앙값 · U · p · $P(\text{FN} < \text{TN})$), `hist_genus.png`, `hist_all.png`, `box_genus.png`


In [ ]:
import os, glob, time
from concurrent.futures import ThreadPoolExecutor
from types import SimpleNamespace
import numpy as np, pandas as pd, torch, h5py
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu
from proteomefit import data, features, calib
from proteomefit.batching import make_batches
from proteomefit.config import Config
from proteomefit.models import FitNet, ds_blocks

cfg = Config(droprate=0.1, n_branch=1000, br_feat=128, br_hidden=(32,), block_p=0.5)   # 04-ablation_fitpred 와 같은 모델 구조
ESS_DIR    = '../results/esscls_ablation/poolesm-noseq'      # 분류 교차검증 결과 (서비스 모델, 보정 확률 pred_calib.npz)
FIT_DIR    = '../results/fitpred_ablation/noseq'             # 적합도 교차검증 결과 (서비스 모델, 폴드별 model.pt)
INPUTS     = ('esm', 'prot', 'bio', 'ctx', 'ctxp', 'ctxb')   # noseq 입력 조합
OUT_DIR    = '../results/fitstat_essfn'
WORK_DIR   = os.path.join(OUT_DIR, 'work')                    # 특성 · 직교상동체 · genus 별 예측 캐시
THRESH     = 0.5                                              # 필수 판정 기준 conf_ess (서비스와 같다)
CHUNK      = 1024                                             # h5 에서 한 번에 연속으로 읽는 단백질 수
MAX_BATCH  = 512                                              # 추론 배치 최대 샘플 수
TOK_BUDGET = 131072                                           # 배치의 (샘플 수 x 최대 토큰 수) 상한 (fp32 토큰 약 0.6 GB)
GENERA_RUN = None                                             # None = 28개 genus 전부, 예: ['Kangiella'] = 빠른 확인
os.makedirs(WORK_DIR, exist_ok=True)

dev = cfg.device
print('device:', dev, (f'| {torch.cuda.get_device_name(0)} {torch.cuda.get_device_properties(0).total_memory / 2**30:.1f} GiB'
                       if dev != 'cpu' else ''))


## 1. 데이터 · 고정 입력

`load_samples(finite_only=False)` 로 **필수 유전자를 포함한** 라벨 있는 샘플 195,354개를 읽는다. 적합도 학습에 쓰인 샘플은 그중 `fit_mean` 이 수치인 177,317개(`fin`)다.
ProteomeLM · Biopython 블록은 03 · 04 노트북과 같게 전 유전자에서 균주별 z-정규화한 뒤 샘플만 고른다. Biopython 특성과 직교상동체 목록은 오래 걸리므로 `work/` 에 저장해 다시 쓴다.

In [ ]:
D   = data.load_samples(cfg.base_dir, cfg.label_col, finite_only=False)    # 라벨 있는 샘플 [n = 195,354] (필수 포함)
y   = data.ess_labels(D)                                                    # 필수 라벨 [n] (1 = 필수)
fin = np.isfinite(D.yb[D.keep])                                             # 적합도 수치 있음 [n] = 회귀 학습 · 평가 샘플

prot_raw, esm_mean_raw = data.load_embeddings(cfg.prot_h5, cfg.mean_h5, D)   # [n_all, 1152] x 2
D.PROT = features.zscore_per_org(prot_raw, D.org_code_all)[D.keep]           # prot 블록 [n, 1152]

bio_path = os.path.join(WORK_DIR, 'bio_all.npy')                            # Biopython 특성 [n_all, 140] (처음 한 번)
if not os.path.exists(bio_path):
    np.save(bio_path, features.bio_features(D.base.seq_aa.fillna('').to_numpy(), D.base.seq_dna.fillna('').to_numpy()))
D.BIO = features.zscore_per_org(np.load(bio_path), D.org_code_all)[D.keep]   # bio 블록 [n, 140]

ort_path = os.path.join(WORK_DIR, 'ortho.npz')                              # 직교상동체 목록 3종 (처음 한 번, GPU)
if not os.path.exists(ort_path):
    O = features.ortholog_lists(esm_mean_raw, prot_raw, D.org_code_all, D.n_org, max(cfg.topk), cfg.sim_chunk, dev)
    np.savez(ort_path, **{f'{s}_{i}': a for s, t in O.items() for i, a in enumerate(t)})
    del O
z = np.load(ort_path)
D.ORT = {s: tuple(z[f'{s}_{i}'] for i in range(3)) for s in ('esm', 'prot', 'both')}   # {공간: (sim, idx, para)}
D.block_dim = features.block_dims(D, cfg.seq_dim)
del prot_raw, esm_mean_raw, z
print(f'samples {len(y):,} (essential {int(y.sum()):,}, with fitness {int(fin.sum()):,}) | blocks', ds_blocks(INPUTS))


## 2. 폴드 구성 · 분류 판정

적합도 모델의 외부 폴드 순서와 내부 검증 genus 는 `FIT_DIR/metrics_total.csv` 에서 그대로 읽는다 (학습 때의 분할).
분류 판정은 시험 genus 의 내부 폴드 3개가 낸 보정 확률 `conf_ess` 의 평균이 0.5 이상이면 필수다.

In [ ]:
tot    = pd.read_csv(os.path.join(FIT_DIR, 'metrics_total.csv'))
GENERA = tot.genus[pd.to_numeric(tot.fold, errors='coerce').notna()].to_numpy()   # 적합도 외부 폴드 순서 [28]
VAL    = calib.val_genera(FIT_DIR)                                            # 시험 genus -> 내부 검증 genus 3개
S      = SimpleNamespace(GENERA=GENERA, gid=pd.Categorical(D.gen, categories=GENERA).codes.astype(int))   # 샘플 -> genus 번호 [n]
assert (S.gid >= 0).all()


def ess_conf(ess_dir, gen, genera):
    # 분류 교차검증 보정 확률 -> 유전자별 conf_ess [n] (시험 genus 의 내부 폴드 3개 평균)
    conf = np.full(len(gen), np.nan)
    for G in genera:
        te = np.flatnonzero(gen == G)                                         # 시험 샘플 (분류 샘플 순서 = 오름차순)
        P  = [np.load(f)['prob'] for f in sorted(glob.glob(os.path.join(ess_dir, f'fold??_{G}', 'inner*', 'pred_calib.npz')))]
        assert len(P) == 3 and all(len(p) == len(te) for p in P), G
        conf[te] = np.mean(P, 0)
    return conf


CONF = ess_conf(ESS_DIR, D.gen, GENERA)                                      # [n]
call = CONF >= THRESH                                                        # 필수 판정 [n]
print(f'essential: TP {int((call & (y == 1)).sum()):,} | FN {int((~call & (y == 1)).sum()):,}   '
      f'non-essential: TN {int((~call & (y == 0)).sum()):,} | FP {int((call & (y == 0)).sum()):,}')


## 3. 추론 함수

- **데이터셋** — `features.build_dataset` 을 필수 포함 샘플에 그대로 쓰고, 표준화 기준 `ref` 만 원래 회귀 훈련 샘플로 준다. 시험 genus 행만 남겨 GPU 로 옮긴다
- **토큰 공급** — 시험 genus 의 균주마다 h5 의 연속 구간을 `CHUNK` 개 단백질씩 읽는다 (HDD 는 연속 읽기가 빠르다). 다음 구간은 백그라운드 스레드가 미리 읽는다
- **배치** — 구간 안에서 토큰 길이순으로 묶고, 핀 메모리 버퍼에 패딩해 GPU 로 비동기 복사한다. 토큰은 한 번만 옮기고 내부 폴드 모델 3개에 같이 넣는다

In [ ]:
def fold_datasets(D, S, blocks, k, vals, fin, te, cfg):
    # 시험 genus k · 검증 genus 번호 3개 · 시험 샘플 te [m] -> 내부 폴드별 데이터셋 [m, d] fp16 x 3 (CPU)
    out = []
    for v in vals:
        ref = np.flatnonzero(fin & (S.gid != k) & (S.gid != v))           # 회귀 훈련 샘플 (학습 때 표준화 기준과 같다)
        M = features.build_dataset(D, S, blocks, k, v, ref, cfg.topk, cfg.t_eval, 'cpu')   # [n, d]
        out.append(M[torch.from_numpy(te)].clone())
        del M
    return out


def load_fit_model(path, inputs, block_dim, cfg, device):
    # 폴드 모델 가중치 -> eval 모드 FitNet (가지별 입력 열 idx 도 state_dict 에 들어 있다)
    net = FitNet(inputs, block_dim, cfg.seed, n_branch=cfg.n_branch, br_feat=cfg.br_feat, br_hidden=cfg.br_hidden,
                 block_p=cfg.block_p, droprate=cfg.droprate, n_head=cfg.n_head)
    net.load_state_dict(torch.load(path, map_location='cpu', weights_only=True))
    return net.to(device).eval()


def org_pieces(org_codes, chunk):
    # 시험 샘플의 균주 번호 [m] (TSV 순) -> 같은 균주 안의 연속 위치 묶음 목록 (각 묶음 <= chunk)
    cut = np.flatnonzero(np.diff(org_codes) != 0) + 1
    return [p for blk in np.split(np.arange(len(org_codes)), cut) for p in np.array_split(blk, -(-len(blk) // chunk))]


def stream_tokens(h5_path, rows, pieces):
    # TSV 행 번호 [m] · 묶음 목록 -> (묶음 위치 [c], 토큰 [t, 1152] fp16, 묶음 안 시작 [c], 길이 [c]) 생성기. 다음 묶음은 미리 읽는다
    with h5py.File(h5_path, 'r') as f, ThreadPoolExecutor(1) as ex:
        emb, off = f['emb'], f['offsets'][:].astype(np.int64)
        def read(p):
            r = rows[p]
            a, z = int(off[r[0]]), int(off[r[-1] + 1])                     # 묶음의 연속 구간 (사이의 라벨 없는 유전자 포함)
            return p, emb[a:z], off[r] - a, off[r + 1] - off[r]
        fut = ex.submit(read, pieces[0])
        for i in range(len(pieces)):
            cur = fut.result()
            if i + 1 < len(pieces):
                fut = ex.submit(read, pieces[i + 1])
            yield cur


def predict_fold(nets, dss, h5_path, rows, pieces, max_batch, tok_budget, device):
    # 모델 · 데이터셋 각 3개 · 시험 샘플 TSV 행 [m] -> 예측 평균 mu [3, m] · 가지 간 표준편차 sd [3, m]
    mu, sd = (np.zeros((len(nets), len(rows)), np.float32) for _ in range(2))
    pin = device != 'cpu'
    buf = torch.empty(tok_budget * 1152, dtype=torch.float16, pin_memory=pin)   # 패딩 토큰 버퍼 (핀 메모리)
    dsg = [d.to(device) for d in dss]                                            # [m, d] fp16 GPU x 3
    with torch.inference_mode():
        for p, slab, st, ln in stream_tokens(h5_path, rows, pieces):
            for b in make_batches(np.arange(len(p)), ln, max_batch, tok_budget):   # 길이순 배치 (묶음 안 위치)
                L = int(ln[b].max())
                x = buf[:len(b) * L * 1152].view(len(b), L, 1152).zero_()
                for r, i in enumerate(b):
                    x[r, :ln[i]] = torch.from_numpy(slab[st[i]:st[i] + ln[i]])
                tok = x.to(device, non_blocking=pin).float()                   # [B, L, 1152] fp32
                pad = torch.arange(L, device=device)[None] >= torch.from_numpy(ln[b]).to(device)[:, None]   # [B, L] True = 패딩
                g = torch.from_numpy(p[b]).to(device)                           # 시험 샘플 안 위치 [B]
                for j, net in enumerate(nets):
                    P = net(tok, pad, None, dsg[j][g])                          # 가지별 예측 [B, 1000]
                    mu[j, p[b]] = P.mean(1).cpu().numpy()                       # .cpu() 가 동기화하므로 버퍼 재사용이 안전하다
                    sd[j, p[b]] = P.std(1).cpu().numpy()
    return mu, sd


## 4. genus 별 추론 · 재현 확인

genus 마다 내부 폴드 데이터셋 3개와 모델 3개를 준비해 시험 샘플 전부(필수 포함)를 예측하고 `work/pred_##_<genus>.npz` 로 저장한다.
재현 확인 — 적합도 수치가 있는 샘플의 예측을 학습 때 저장한 `pred.npz` 와 비교한다 (`r` = Pearson, `max_abs_diff` = 최대 절대 차이).
학습은 bf16, 여기는 fp32 이므로 완전히 같지는 않지만 `r` 이 1 에 가까워야 한다.

In [ ]:
MU = np.full((3, len(y)), np.nan, np.float32); SD = MU.copy()            # 내부 폴드 모델별 예측 [3, n]
check, t_all = [], time.perf_counter()
for k, G in enumerate(GENERA):
    if GENERA_RUN and G not in GENERA_RUN:
        continue
    te, fdir = np.flatnonzero(S.gid == k), os.path.join(FIT_DIR, f'fold{k + 1:02d}_{G}')
    cache, t0 = os.path.join(WORK_DIR, f'pred_{k + 1:02d}_{G}.npz'), time.perf_counter()
    if not os.path.exists(cache):
        vals = [int(np.flatnonzero(GENERA == v)[0]) for v in VAL[G]]
        dss  = fold_datasets(D, S, ds_blocks(INPUTS), k, vals, fin, te, cfg)
        nets = [load_fit_model(os.path.join(fdir, f'inner{j + 1:02d}', 'model.pt'), INPUTS, D.block_dim, cfg, dev) for j in range(3)]
        rows = D.keep[te]                                                       # 시험 샘플의 TSV(= h5) 행 [m]
        mu, sd = predict_fold(nets, dss, cfg.esm_h5, rows, org_pieces(D.org_code_all[rows], CHUNK), MAX_BATCH, TOK_BUDGET, dev)
        np.savez(cache, te=te, mu=mu, sd=sd)
        del nets, dss
        if dev != 'cpu':
            torch.cuda.empty_cache()
    z = np.load(cache)
    MU[:, te], SD[:, te] = z['mu'], z['sd']
    old = np.stack([np.load(os.path.join(fdir, f'inner{j + 1:02d}', 'pred.npz'))['mu'] for j in range(3)])   # 학습 때 예측 [3, m_fin]
    new = z['mu'][:, fin[te]]                                                # 이번 예측 중 적합도 수치가 있는 샘플 [3, m_fin]
    d = np.abs(old - new)                                                    # |학습 때 - 이번| [3, m_fin]
    check.append(dict(fold=k + 1, genus=G, n=len(te), n_ess=int(y[te].sum()), r=np.corrcoef(old.ravel(), new.ravel())[0, 1],
                      median_abs_diff=float(np.median(d)), p99_abs_diff=float(np.percentile(d, 99)), max_abs_diff=float(d.max()),
                      sec=round(time.perf_counter() - t0, 1)))
    c = check[-1]
    print(f"fold{k + 1:02d} {G:<16} n {c['n']:>6,} (ess {c['n_ess']:>4}) | reproduce r {c['r']:.5f} |diff| median "
          f"{c['median_abs_diff']:.4f} p99 {c['p99_abs_diff']:.4f} max {c['max_abs_diff']:.3f} | {c['sec']:.0f}s")
check = pd.DataFrame(check)
check.to_csv(os.path.join(OUT_DIR, 'check.csv'), index=False)
print(f'total {time.perf_counter() - t_all:.0f}s | reproduce r min {check.r.min():.5f}')


## 5. 그룹 · 통계

유전자별 예측 적합도 = 내부 폴드 모델 3개의 $\mu$ 평균. FN 필수 · TN 비필수 두 그룹을 genus 별 · 전체로 Mann–Whitney U 검정(양측)으로 비교한다.
- $U_{\text{FN}}$ = FN 값이 TN 값보다 큰 (FN, TN) 쌍의 수 (동점은 0.5), $P(\text{FN} < \text{TN}) = 1 - U_{\text{FN}} / (n_{\text{FN}}\, n_{\text{TN}})$
- 예: FN 3개 (−3, −1, 0.5) · TN 2개 (0, 1) 이면 쌍 6개 중 FN 이 큰 쌍은 (0.5, 0) 1개 → $U_{\text{FN}} = 1$, $P(\text{FN} < \text{TN}) = 1 - 1/6 = 0.83$

In [ ]:
done = ~np.isnan(MU[0])                                                    # 추론한 샘플 [n]
mu   = np.full(len(y), np.nan); mu[done] = MU[:, done].mean(0)               # 유전자별 예측 적합도 = 모델 3개 평균 [n]
sd   = np.full(len(y), np.nan); sd[done] = SD[:, done].mean(0)               # 가지 간 표준편차의 모델 3개 평균 [n]
FN   = done & (y == 1) & ~call                                               # 비필수로 오분류된 필수 유전자
TN   = done & (y == 0) & ~call                                               # 비필수로 맞게 분류된 비필수 유전자


def mwu(a, b):
    # 두 표본 [n1] · [n2] -> (U, p, P(a < b)) Mann-Whitney U 검정 (양측). U = a 가 b 보다 큰 쌍의 수 (동점 0.5), 표본이 없으면 NaN
    if min(len(a), len(b)) == 0:
        return np.nan, np.nan, np.nan
    U, p = mannwhitneyu(a, b, alternative='two-sided')
    return U, p, 1 - U / (len(a) * len(b))


def group_stats(mu, fn, tn, gen, genera):
    # 예측 적합도 · 두 그룹 마스크 [n] -> genus 별 + 전체 통계 표 (평균 · 중앙값 · U · p · P(FN < TN))
    rows = []
    for G, m in [(G, gen == G) for G in genera] + [('all genera', np.ones(len(gen), bool))]:
        a, b = mu[fn & m], mu[tn & m]
        U, p, p_lt = mwu(a, b)
        stat = lambda v, f: f(v) if len(v) else np.nan
        rows.append(dict(genus=G, n_fn=len(a), n_tn=len(b), mean_fn=stat(a, np.mean), mean_tn=stat(b, np.mean),
                         median_fn=stat(a, np.median), median_tn=stat(b, np.median), U=U, p=p, p_fn_lt_tn=p_lt))
    return pd.DataFrame(rows)


ST = group_stats(mu, FN, TN, D.gen, [G for G in GENERA if done[D.gen == G].any()])
ST.to_csv(os.path.join(OUT_DIR, 'stats.csv'), index=False, float_format='%.6g')   # p 는 반올림하지 않는다 (아주 작다)
grp = np.select([FN, TN, done & (y == 1) & call, done & (y == 0) & call], ['FN', 'TN', 'TP', 'FP'], '')
pd.DataFrame({'org_id': D.lab.org_id, 'gene_id': D.lab.gene_id, 'genus': D.gen, 'essential': y, 'conf_ess': CONF.round(6),
              'call': np.where(call, 'essential', 'non-essential'), 'group': grp, 'mu': mu.round(6), 'sd': sd.round(6)}
             )[done].to_csv(os.path.join(OUT_DIR, 'pred_all.tsv'), sep='\t', index=False)
f3 = '{:.3f}'.format
print(ST.to_string(index=False, formatters={'mean_fn': f3, 'mean_tn': f3, 'median_fn': f3, 'median_tn': f3,
                                            'U': '{:.0f}'.format, 'p': '{:.1e}'.format, 'p_fn_lt_tn': f3}))


## 6. 그림

- `hist_genus.png` — genus 별 패널: 두 분포를 겹친 히스토그램 (y 축 로그, 투명도 0.3, 평균 = 같은 색 점선, 제목 아래 평균 · 중앙값 · Mann–Whitney U p · $P(\text{FN} < \text{TN})$)
- `hist_all.png` — 전체 genus 를 합친 같은 그림
- `box_genus.png` — genus 별 두 분포의 박스플롯. 두 박스를 잇는 선 위에 Mann–Whitney U 검정 유의성 별표, 그 위 세로 글씨 = 박스별 유전자 수, 범례 = 그림 아래
- 히스토그램 구간은 모든 패널이 같고, 0.05 – 99.95 백분위 밖의 값은 양 끝 구간에 넣는다

In [ ]:
C_FN, C_TN = 'tab:blue', 'tab:green'
LAB_FN, LAB_TN = 'essential called non-essential (FN)', 'non-essential called non-essential (TN)'


def stars(p):
    # p 값 -> 유의성 표시
    return 'n/a' if not np.isfinite(p) else '***' if p < 1e-3 else '**' if p < 1e-2 else '*' if p < 0.05 else 'ns'


def hist_panel(ax, a, b, bins, title):
    # FN (a) · TN (b) 예측 적합도 -> 겹친 히스토그램 (log y, 투명도 0.3, 평균 점선)
    lo, hi = bins[0], bins[-1]
    for v, c in ((b, C_TN), (a, C_FN)):                                      # TN 먼저, 적은 FN 을 위에
        if len(v):
            ax.hist(np.clip(v, lo, hi), bins=bins, color=c, alpha=0.3)
            ax.axvline(v.mean(), color=c, ls='--', lw=1.2)
    ax.set_yscale('log'); ax.set_xlim(lo, hi); ax.set_title(title, fontsize=8)


def stat_title(name, r):
    # 통계 표 1줄 -> 패널 제목 (이름 / FN · TN 평균 · 중앙값 / Mann-Whitney U p · P(FN < TN))
    fn = f"FN mean {r.mean_fn:.2f}, median {r.median_fn:.2f} (n={int(r.n_fn):,})" if r.n_fn else 'FN none'
    tn = f"TN mean {r.mean_tn:.2f}, median {r.median_tn:.2f} (n={int(r.n_tn):,})"
    p_txt = 'p < 1e-300' if r.p == 0 else f'p = {r.p:.1e}'                    # p = 0 은 배정밀도 아래로 내려간 값
    test = (f"Mann-Whitney U {p_txt} {stars(r.p)}, P(FN<TN) = {r.p_fn_lt_tn:.2f}" if np.isfinite(r.p)
            else 'Mann-Whitney U n/a')
    return f"{name}\n{fn}\n{tn}\n{test}"


v_all = mu[FN | TN]
bins  = np.linspace(*np.percentile(v_all, [0.05, 99.95]), 61)               # 공통 구간 60개
legend = [plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.3) for c in (C_FN, C_TN)] + \
         [plt.Line2D([], [], color='0.3', ls='--')]
leg_lab = [LAB_FN, LAB_TN, 'group mean']
xlab = 'predicted knockout fitness (mu, within-strain z)'

# genus 별 패널
Gs = sorted(ST.genus[ST.genus != 'all genera'])                            # 그림의 genus 순서 = 알파벳 순
nc = 5; nr = -(-len(Gs) // nc)
fig, axes = plt.subplots(nr, nc, figsize=(4.2 * nc, 3.8 * nr), squeeze=False)
for ax, G in zip(axes.flat, Gs):
    m = D.gen == G
    hist_panel(ax, mu[FN & m], mu[TN & m], bins, stat_title(G, ST.set_index('genus').loc[G]))
for ax in axes.flat[len(Gs):]:
    ax.axis('off')
for ax in axes[-1]:
    ax.set_xlabel(xlab, fontsize=8)
for ax in axes[:, 0]:
    ax.set_ylabel('genes (log)', fontsize=8)
fig.legend(legend, leg_lab, loc='lower center', ncol=3, fontsize=9, frameon=False)
fig.suptitle('Predicted fitness of essential genes missed by the classifier vs correctly called non-essential genes (held-out genera)', fontsize=11)
fig.tight_layout(rect=(0, 0.03, 1, 0.98)); fig.savefig(os.path.join(OUT_DIR, 'hist_genus.png'), dpi=130); plt.show()

# 전체 genus 합침
fig, ax = plt.subplots(figsize=(7, 5.1))
hist_panel(ax, mu[FN], mu[TN], bins, stat_title(f'all genera ({len(Gs)})', ST.set_index('genus').loc['all genera']))
ax.set_xlabel(xlab); ax.set_ylabel('genes (log)'); ax.legend(legend, leg_lab, fontsize=8)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, 'hist_all.png'), dpi=150); plt.show()

# genus 별 박스플롯 + 두 박스를 잇는 선 위 유의성
def bracket(ax, x1, x2, y, h, text):
    # 두 박스 위치 x1 · x2 · 선 높이 y · 다리 길이 h -> 연결선 (양 끝이 아래로 꺾임) + 선 위 유의성 표시
    ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], color='k', lw=0.8)
    ax.text((x1 + x2) / 2, y + h, text, ha='center', va='bottom', fontsize=9)


span = np.ptp(v_all)                                                         # 세로 범위 (연결선 · 여백 기준)
fig, ax = plt.subplots(figsize=(max(10, 0.62 * len(Gs)), 7.2))
for i, G in enumerate(Gs):
    m = D.gen == G
    a, b = mu[FN & m], mu[TN & m]
    for v, c, x in ((a, C_FN, i - 0.19), (b, C_TN, i + 0.19)):
        if len(v):
            bp = ax.boxplot(v, positions=[x], widths=0.32, patch_artist=True, showfliers=True,
                            flierprops=dict(marker='.', markersize=1.5, alpha=0.3, markeredgecolor=c), medianprops=dict(color='k'))
            bp['boxes'][0].set(facecolor=c, alpha=0.3, edgecolor=c)
    top = max(v.max() for v in (a, b) if len(v)) + 0.03 * span                  # 두 분포의 최댓값 (이상치 포함) 바로 위
    if len(a):
        bracket(ax, i - 0.19, i + 0.19, top, 0.02 * span, stars(ST.set_index('genus').loc[G, 'p']))
    else:
        ax.text(i, top + 0.02 * span, 'no FN', ha='center', va='bottom', fontsize=9)   # genus 구간 가운데, 유의성 표시와 같은 높이
    for v, x in ((a, i - 0.19), (b, i + 0.19)):                               # 박스별 유전자 수 = 유의성 표시 바로 위 세로 글씨
        if len(v):
            ax.annotate(f'n={len(v):,}', (x, top + 0.02 * span), xytext=(0, 20), textcoords='offset points',
                        rotation=90, ha='center', va='bottom', fontsize=8.5)
ax.set_xlim(-0.6, len(Gs) - 0.4)
ax.set_ylim(v_all.min() - 0.03 * span, v_all.max() + 0.33 * span)           # 위 = 연결선 · 유의성 · n 표기 여백
ax.set_xticks(range(len(Gs)), Gs, rotation=45, ha='right', rotation_mode='anchor', fontsize=12)
ax.axhline(0, color='0.6', lw=0.8, ls=':')
ax.set_ylabel(xlab)
fig.legend([plt.Rectangle((0, 0), 1, 1, color=c, alpha=0.3) for c in (C_FN, C_TN)], [LAB_FN, LAB_TN],
           loc='lower center', ncol=2, fontsize=10, frameon=False)            # 그림 아래 가운데 한 줄
ax.set_title('Mann-Whitney U test per genus (two-sided): *** p < 0.001, ** p < 0.01, * p < 0.05, ns = not significant', fontsize=9)
fig.tight_layout(rect=(0, 0.05, 1, 1)); fig.savefig(os.path.join(OUT_DIR, 'box_genus.png'), dpi=150); plt.show()
